Hugging Face transformers 라이브러리를 사용하여 문서 요약 모델을 구현하는 미션입니다. 데이터 로드 및 전처리부터 요약 모델 실행, 결과 평가까지 전체 파이프라인을 구축해 보세요.

In [1]:
import os

ROOT_DIR = os.getcwd()

# 코랩 모드
if ROOT_DIR == "/content":
    pass
    # print("[[ colab ]]")
    
    # import unicodedata
    
    # DATA_DIR = os.path.join(ROOT_DIR, "data")
    # TEXT_DIR = os.path.join(ROOT_DIR, "raw")

    # if "raw.tar.gz" not in os.listdir():
    #     # subprocess()
    #     !wget https://github.com/wonbywondev/ML-DL/releases/download/data-v3/raw.tar.gz
    # else:
    #     print("· raw.tar.gz (O)")


    # if not os.path.exists(TEXT_DIR):
    #     # !tar -xzvf raw.tar.gz -C /content
    # else:
    #     print("· data (O)")


    # if not os.path.exists(DATA_DIR):
    #     os.mkdir(DATA_DIR)


    # train_json_path = os.path.join(TEXT_DIR, "일상생활및구어체_한영_train_set.json")
    # val_json_path = os.path.join(TEXT_DIR, "일상생활및구어체_한영_valid_set.json")

    # train_json_path = unicodedata.normalize("NFC", train_json_path)
    # val_json_path = unicodedata.normalize("NFC", val_json_path)

# 로컬 모드
else:
    print("[[ local ]]")
    ROOT_DIR = "/".join(ROOT_DIR.split("/")[:-1])
    DATA_DIR = os.path.join(ROOT_DIR, "data")
    RAW_DIR = os.path.join(DATA_DIR, "raw")

    train_edit_json_path = os.path.join(RAW_DIR, "train_original_editorial.json")
    train_law_json_path = os.path.join(RAW_DIR, "train_original_news.json")
    train_news_json_path = os.path.join(RAW_DIR, "train_original_law.json")
    val_edit_json_path = os.path.join(RAW_DIR, "valid_original_editorial.json")
    val_law_json_path = os.path.join(RAW_DIR, "valid_original_news.json")
    val_news_json_path = os.path.join(RAW_DIR, "valid_original_law.json")

[[ local ]]


In [2]:
RAW_DIR

'/Users/won/dev/00_codeit/0_mission/12_DL_transformers/data/raw'

In [3]:
# 기타 환경 설정
import torch
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import gc


# 시각화 관련 설정
try:
    plt.rcParams['font.family'] = 'Apple SD Gothic Neo'
except:
    try:
        plt.rcParams['font.family'] = 'NanumGothic'
    except:
        plt.rcParams['font.family'] = 'AppleGothic'

plt.rcParams['axes.unicode_minus'] = False
fm._load_fontmanager(try_read_cache=False)


# 디바이스 설정
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps") # 맥 GPU
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda") # 윈도우 GPU
else:
    DEVICE = torch.device("cpu") # CPU


# 캐시 지우기 함수 생성
def clean_cache():
    gc.collect()
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()


# MallocStackLogging 에러 출력 방지
os.environ.pop("MallocStackLogging", None)
os.environ.pop("MallocStackLoggingNoCompact", None)
os.environ.pop("DYLD_INSERT_LIBRARIES", None)

Matplotlib is building the font cache; this may take a moment.


In [4]:
import json

with open(train_edit_json_path, "r", encoding="utf-8") as f:
    raw_train_edit = json.load(f)

In [5]:
raw_train_edit.keys()

dict_keys(['name', 'delivery_date', 'documents'])

In [6]:
from collections.abc import Mapping, Sequence
from pathlib import Path

def dig(data, path="root", level=0, seen=None):
    if seen is None:
        seen = set()
    indent = "  " * level
    if isinstance(data, (Mapping, Sequence)) and not isinstance(data, (str, bytes, bytearray)):
        obj_id = id(data)
        if obj_id in seen:
            print(f"{indent}{path}: <cycle>")
            return
        seen.add(obj_id)

    if isinstance(data, Mapping):
        print(f"{indent}{path}: dict ({len(data)})")
        for key, value in data.items():
            next_path = f"{path}.{key}" if path else str(key)
            dig(value, next_path, level + 1, seen)
    elif isinstance(data, Sequence) and not isinstance(data, (str, bytes, bytearray)):
        print(f"{indent}{path}: list ({len(data)})")
        if data:
            dig(data[0], f"{path}[0]", level + 1, seen)
        else:
            print(f"{indent}  {path}[empty]: -> empty")
    else:
        print(f"{indent}{path}: -> {type(data).__name__}")

raw_train_edit = json.loads(Path(train_edit_json_path).read_text())
dig(raw_train_edit)


root: dict (3)
  root.name: -> str
  root.delivery_date: -> str
  root.documents: list (56760)
    root.documents[0]: dict (14)
      root.documents[0].id: -> str
      root.documents[0].category: -> str
      root.documents[0].media_type: -> str
      root.documents[0].media_sub_type: -> str
      root.documents[0].media_name: -> str
      root.documents[0].size: -> str
      root.documents[0].char_count: -> str
      root.documents[0].publish_date: -> str
      root.documents[0].title: -> str
      root.documents[0].text: list (5)
        root.documents[0].text[0]: list (4)
          root.documents[0].text[0][0]: dict (3)
            root.documents[0].text[0][0].index: -> int
            root.documents[0].text[0][0].sentence: -> str
            root.documents[0].text[0][0].highlight_indices: -> str
      root.documents[0].annotator_id: -> int
      root.documents[0].document_quality_scores: dict (4)
        root.documents[0].document_quality_scores.readable: -> int
        root.docum

In [ ]:
from collections.abc import Mapping, Sequence

def iter_leaf_paths(node, prefix=()):
    if isinstance(node, Mapping):
        if not node:
            yield prefix, None
        else:
            for key, value in node.items():
                yield from iter_leaf_paths(value, prefix + (key,))
    elif isinstance(node, Sequence) and not isinstance(node, (str, bytes, bytearray)):
        if not node:
            yield prefix + ('__empty__',), None
        else:
            for idx, value in enumerate(node):
                yield from iter_leaf_paths(value, prefix + (idx,))
    else:
        yield prefix, node

def flatten_document(document, joiner=".", keep_keys=None):
    keep_keys = keep_keys or set()
    flattened = {}
    for key in keep_keys:
        if key in document:
            flattened[key] = document[key]  # text 등은 통째로 저장
    for path, value in iter_leaf_paths(document):
        if path and path[0] in keep_keys:
            continue  # text.* 경로는 펼치지 않음
        flattened[joiner.join(map(str, path))] = value
    return flattened

KEEP_AS_IS = {"text"}
train_edit_meta = {f"dataset.{k}": v for k, v in raw_train_edit.items() if k != "documents"}
train_edit_rows = [
    {**train_edit_meta, **flatten_document(doc, keep_keys=KEEP_AS_IS)}
    for doc in raw_train_edit["documents"]
    ]

In [26]:
import pandas as pd

train_edit_df = pd.DataFrame(train_edit_rows)

train_edit_df.columns

Index(['dataset.name', 'dataset.delivery_date', 'text', 'id', 'category',
       'media_type', 'media_sub_type', 'media_name', 'size', 'char_count',
       'publish_date', 'title', 'annotator_id',
       'document_quality_scores.readable', 'document_quality_scores.accurate',
       'document_quality_scores.informative',
       'document_quality_scores.trustworthy', 'extractive.0', 'extractive.1',
       'extractive.2', 'abstractive.0', 'drop_char_count'],
      dtype='object')

In [37]:
train_edit_df.head(1)

,dataset.name,dataset.delivery_date,text,id,category,media_type,media_sub_type,media_name,size,char_count,...,annotator_id,document_quality_scores.readable,document_quality_scores.accurate,document_quality_scores.informative,document_quality_scores.trustworthy,extractive.0,extractive.1,extractive.2,abstractive.0,drop_char_count
0,사설/잡지 문서 프로젝트,2020-12-23 15:00:19,"[[{'index': 0, 'sentence': '이명박 대통령이 어제 30대 그룹...",100062073,오피니언,online,경제지,매일경제,medium,1153,...,3924,4,3,3,3,0,6,7,이명박 대통령은 어제 30대 그룹 총수를 모아놓고 시대적 요구는 역시 총수가 앞장서...,NaN


In [36]:
column_list = [column for column in train_edit_df.columns if column != "text"]

sum(train_edit_df[column_list].duplicated())

0

In [38]:
train_edit_df.iloc[1]["text"][0]

[{'index': 0,
  'sentence': '이명박 정부의 첫 대통령실장을 지낸 류우익 주중대사가 통일부 장관에 내정되면서 대북 정책에 변화가 예상된다.',
  'highlight_indices': '8,9'},
 {'index': 1,
  'sentence': '청와대는 "지금까지의 통일 정책 일관성을 유지해갈 것"이라고 강조했지만, 현인택 장관 경질은 해임건의안을 제출한 야당 측 요구를 의식한 측면이 크다.',
  'highlight_indices': ''},
 {'index': 2,
  'sentence': '이 대통령의 인사 스타일에 비춰보더라도 기존 정책 기조 유지가 목표라면 2년7개월째 자리를 지킨 현 장관을 교체할 이유가 없었을 것이다.',
  'highlight_indices': '0,1;54,55'}]

In [39]:
from collections.abc import Mapping, Sequence
from pathlib import Path
import json

def iter_leaves(node, prefix=()):
    if isinstance(node, Mapping):
        if not node:
            yield prefix, None
        else:
            for key, value in node.items():
                yield from iter_leaves(value, prefix + (key,))
    elif isinstance(node, Sequence) and not isinstance(node, (str, bytes, bytearray)):
        if not node:
            yield prefix + ('__empty__',), None
        else:
            for idx, value in enumerate(node):
                yield from iter_leaves(value, prefix + (idx,))
    else:
        yield prefix, node

def flatten(node):
    return {".".join(map(str, path)): value for path, value in iter_leaves(node)}

raw = json.loads(Path(train_edit_json_path).read_text())
meta = {f"dataset.{k}": v for k, v in raw.items() if k != "documents"}
rows = [{**meta, **flatten(doc)} for doc in raw["documents"]]


In [63]:
import pandas as pd

train_edit_df = pd.DataFrame(rows)

for i in range(1, 45):
    mask = train_edit_df[f"text.{i}.0.sentence"].notna()
    dupes = train_edit_df.loc[mask, f"text.{i}.0.sentence"].duplicated(keep=False)
    result = train_edit_df[mask & dupes]

    if len(result) > 0:
        print(f"i: {i} ({len(result)}개)")
        print(result[f"text.{i}.0.sentence"])

i: 1 (7876개)
0        최근 미국 프랑스 벨기에 등에서 부유세가 거론되고 독일조차 2년간 한시적으로 5%의...
3        최근 미국 프랑스 벨기에 등에서 부유세가 거론되고 독일조차 2년간 한시적으로 5%의...
5        물가가 이렇게 뜀박질한 것은 2008년 8월(5.6%) 이래 36개월 만인데 당시는...
6        본회의 표결을 지켜 보려 참석했던 방청객과 기자들은 박희태 의장의 개의선언과 함께 ...
7                  그러나 법원 결정에도 불구하고 반대단체들은 결코 물러설 기세가 아니다.
                               ...                        
56714       우리 경제는 수출이 성장에 미치는 비중이 70% 이상으로, 수출로 먹고사는 구조다.
56715    일본 정부는 그동안 이 품목들의 한국 수출 절차를 간소화하는 우대 조치를 취해왔으나...
56751       우리 경제는 수출이 성장에 미치는 비중이 70% 이상으로, 수출로 먹고사는 구조다.
56752    일본 정부는 그동안 이 품목들의 한국 수출 절차를 간소화하는 우대 조치를 취해왔으나...
56753    우선 여야는 현행 3 개월인 탄력근로제 단위 기간을 확대하는 내용을 담은 관련 법 ...
Name: text.1.0.sentence, Length: 7876, dtype: object
i: 2 (7904개)
0        그런 면에서 재계에 적절한 방안 마련을 맡기고 정치권이나 여론은 너무 압박하지 말고...
3        그런 면에서 재계에 적절한 방안 마련을 맡기고 정치권이나 여론은 너무 압박하지 말고...
5        지난달 물가는 집중호우 등에 따른 농축수산물 가격 급등(13.3%)에 큰 타격을 입었다.
6        더구나 표결에 앞서 작년까지 국회의장을 지낸 김형오 의원은 "죄 없는 사람이 이 여...
7        3일에는 서울에서 '평화비행기'를 타고 강정마을을

In [44]:
for i in range(1, 45):
    print(sum(train_edit_df[f"text.{i}.0.sentence"].notna().duplicated()))

56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
56758
